In [1]:
import numpy as np
from pathlib import Path
import fastplotlib as fpl
import pandas as pd
from decord import VideoReader
from abc import ABC, abstractmethod
from typing import *
import os

Unable to find extension: VK_EXT_physical_device_drm


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),AMD Radeon RX 570 Series (RADV POLARIS10),DiscreteGPU,Vulkan,Mesa 22.3.6
✅,NVIDIA GeForce RTX 3080,DiscreteGPU,Vulkan,575.57.08
❗ limited,"llvmpipe (LLVM 15.0.6, 256 bits)",CPU,Vulkan,Mesa 22.3.6 (LLVM 15.0.6)
❌,"AMD Radeon RX 570 Series (polaris10, LLVM 15.0.6, DRM 3.49, 6.1.0-41-amd64)",Unknown,OpenGL,4.6 (Core Profile) Mesa 22.3.6


pygfx version from git (0.9.0) and __version__ (0.15.0) don't match.
To silence this warning, use a fully namespaced name.


In [2]:
# decord uses up all the RAM otherwise
os.environ["DECORD_EOF_RETRY_MAX"] = "128"

In [3]:
# Some stuff I copied from mesmerize-core for lazy-loading the video files using decord

slice_or_int_or_range = Union[int, slice, range]

class LazyArray(ABC):
    """
    Base class for arrays that exhibit lazy computation upon indexing
    """

    @property
    @abstractmethod
    def dtype(self) -> str:
        """
        str
            data type
        """
        pass

    @property
    @abstractmethod
    def shape(self) -> Tuple[int, int, int]:
        """
        Tuple[int]
            (n_frames, dims_x, dims_y)
        """
        pass

    @property
    @abstractmethod
    def min(self) -> float:
        """
        float
            min value of the array if it were fully computed
        """
        pass

    @property
    @abstractmethod
    def max(self) -> float:
        """
        float
            max value of the array if it were fully computed
        """
        pass

    @property
    def ndim(self) -> int:
        """
        int
            Number of dimensions
        """
        return len(self.shape)

    @property
    def nbytes(self) -> int:
        """
        int
            number of bytes for the array if it were fully computed
        """
        return np.prod(self.shape + (np.dtype(self.dtype).itemsize,), dtype=np.int64)

    @property
    def nbytes_gb(self) -> float:
        """
        float
            number of gigabytes for the array if it were fully computed
        """
        return self.nbytes / 1e9

    @abstractmethod
    def _compute_at_indices(self, indices: Union[int, slice]) -> np.ndarray:
        """
        Lazy computation logic goes here. Computes the array at the desired indices.

        Parameters
        ----------
        indices: Union[int, slice]
            the user's desired slice, i.e. slice object or int passed from `__getitem__()`

        Returns
        -------
        np.ndarray
            array at the indexed slice
        """
        pass

    def __getitem__(self, item: Union[int, Tuple[slice_or_int_or_range]]):
        if isinstance(item, int):
            indexer = item

        # numpy int scaler
        elif isinstance(item, np.integer):
            indexer = item.item()

        # treat slice and range the same
        elif isinstance(item, (slice, range)):
            indexer = item

        elif isinstance(item, tuple):
            if len(item) > len(self.shape):
                raise IndexError(
                    f"Cannot index more dimensions than exist in the array. "
                    f"You have tried to index with <{len(item)}> dimensions, "
                    f"only <{len(self.shape)}> dimensions exist in the array"
                )

            indexer = item[0]

        else:
            raise IndexError(
                f"You can index LazyArrays only using slice, int, or tuple of slice and int, "
                f"you have passed a: <{type(item)}>"
            )

        # treat slice and range the same
        if isinstance(indexer, (slice, range)):
            start = indexer.start
            stop = indexer.stop
            step = indexer.step

            if start is not None:
                if start > self.n_frames:
                    raise IndexError(
                        f"Cannot index beyond `n_frames`.\n"
                        f"Desired frame start index of <{start}> "
                        f"lies beyond `n_frames` <{self.n_frames}>"
                    )
            if stop is not None:
                if stop > self.n_frames:
                    raise IndexError(
                        f"Cannot index beyond `n_frames`.\n"
                        f"Desired frame stop index of <{stop}> "
                        f"lies beyond `n_frames` <{self.n_frames}>"
                    )

            if step is None:
                step = 1

            # convert indexer to slice if it was a range, allows things like decord.VideoReader slicing
            indexer = slice(start, stop, step)  # in case it was a range object

            # dimension_0 is always time
            frames = self._compute_at_indices(indexer)

            # index the remaining dims after lazy computing the frame(s)
            if isinstance(item, tuple):
                if len(item) == 2:
                    return frames[:, item[1]]
                elif len(item) == 3:
                    return frames[:, item[1], item[2]]

            else:
                return frames

        elif isinstance(indexer, (int, np.integer)):
            return self._compute_at_indices(indexer)

    def __repr__(self):
        return (
            f"{self.__class__.__name__} @{hex(id(self))}\n"
            f"{self.__class__.__doc__}\n"
            f"Frames are computed only upon indexing\n"
            f"shape [frames, x, y]: {self.shape}\n"
        )


class LazyVideo(LazyArray):
    def __init__(
        self,
        path: Union[Path, str],
        min_max: Tuple[int, int] = None,
        **kwargs,
    ):
        """
        LazyVideo reader, basically just a wrapper for ``decord.VideoReader``.
        Should support opening anything that decord can open.

        **Important:** requires ``decord`` to be installed: https://github.com/dmlc/decord

        Parameters
        ----------
        path: Path or str
            path to video file

        min_max: Tuple[int, int], optional
            min and max vals of the entire video, uses min and max of 10th frame if not provided

        as_grayscale: bool, optional
            return grayscale frames upon slicing

        rgb_weights: Tuple[float, float, float], optional
            (r, g, b) weights used for grayscale conversion if ``as_graycale`` is ``True``.
            default is (0.299, 0.587, 0.114)

        kwargs
            passed to ``decord.VideoReader``

        Examples
        --------

        Lazy loading with CPU

        .. code-block:: python

            from mesmerize_core.arrays import LazyVideo

            vid = LazyVideo("path/to/video.mp4")

            # use fpl to visualize

            import fastplotlib as fpl

            iw = fpl.ImageWidget(vid)
            iw.show()


        Lazy loading with GPU, decord must be compiled with CUDA options to use this

        .. code-block:: python

            from decord import gpu
            from mesmerize_core.arrays import LazyVideo

            gpu_context = gpu(0)

            vid = LazyVideo("path/to/video.mp4", ctx=gpu_context)

        """
        self._video_reader = VideoReader(str(path), **kwargs)

        try:
            frame0 = self._video_reader[10].asnumpy()
            self._video_reader.seek(0)
        except IndexError:
            frame0 = self._video_reader[0].asnumpy()
            self._video_reader.seek(0)

        self._shape = (self._video_reader._num_frame, *frame0.shape)

        self._dtype = frame0.dtype

        if min_max is not None:
            self._min, self._max = min_max
        else:
            self._min = frame0.min()
            self._max = frame0.max()

    @property
    def dtype(self) -> str:
        return self._dtype

    @property
    def shape(self) -> Tuple[int, int, int]:
        """[n_frames, x, y, 3 | 4]"""
        return self._shape

    @property
    def min(self) -> float:
        warn("min not implemented for LazyTiff, returning min of 0th index")
        return self._min

    @property
    def max(self) -> float:
        warn("max not implemented for LazyTiff, returning min of 0th index")
        return self._max

    def _compute_at_indices(self, indices: Union[int, slice]) -> np.ndarray:
        a = self._video_reader[indices].asnumpy()
        self._video_reader.seek(0)
        return a

In [4]:
# path with all the files for this session
parent_path = Path("/home/kushal/data/kcenia/")

In [5]:
# load video file as a lazy array
vid = LazyVideo(parent_path.joinpath("mouse1.mp4"))

In [6]:
vid.shape

(200269, 1024, 1280, 3)

### Indexing stuff

In [7]:
vid_indexing = np.load(parent_path.joinpath("camera_times.npy"))
vid_indexing

array([2.57920000e+00, 2.60480000e+00, 2.63040000e+00, ...,
       5.13430933e+03, 5.13433497e+03, 5.13436061e+03], shape=(200269,))

## Used for mapping from reference units (i.e. seconds) to array index

In [8]:
def vid_second_to_index(s: float) -> int:
    return vid_indexing.searchsorted(s)

def tracks_second_to_index(s: float) -> int:
    return df_tracks["times"].searchsorted(s)

In [9]:
df_tracks = pd.read_csv(parent_path.joinpath("video_data.csv"))
df_tracks

,nose_tip_x,nose_tip_y,nose_tip_likelihood,pupil_top_r_x,pupil_top_r_y,pupil_top_r_likelihood,pupil_top_r_x_ens_median,pupil_top_r_y_ens_median,pupil_top_r_x_ens_var,pupil_top_r_y_ens_var,...,tube_bottom_x,tube_bottom_y,tube_bottom_likelihood,tongue_end_l_x,tongue_end_l_y,tongue_end_l_likelihood,tongue_end_r_x,tongue_end_r_y,tongue_end_r_likelihood,times
0,167.842781,286.913076,0.999864,443.894814,190.173240,0.998643,444.052498,190.548458,0.113807,0.075323,...,293.920395,445.772614,0.999971,281.745491,410.380905,0.001576,283.634926,410.398018,0.001351,2.579200
1,167.842781,286.913076,0.999864,443.894966,190.173650,0.998643,444.052498,190.548458,0.113807,0.075323,...,293.920395,445.772614,0.999971,281.745491,410.380905,0.001576,283.634926,410.398018,0.001351,2.604800
2,167.842781,286.913076,0.999864,443.915987,190.191422,0.998643,444.052498,190.548458,0.113807,0.075323,...,293.920395,445.772614,0.999971,281.745491,410.380905,0.001576,283.634926,410.398018,0.001351,2.630400
3,162.909931,293.424446,0.997724,443.963149,190.231734,0.999108,443.924583,190.486732,0.095225,0.068660,...,293.660896,446.639214,0.999071,282.049431,411.219681,0.001381,284.632889,411.008743,0.001241,2.656000
4,164.736584,291.756172,0.993336,444.056465,190.301728,0.998958,443.986042,190.434261,0.106825,0.068595,...,293.693123,446.294495,0.999884,281.405090,410.951775,0.001358,283.811623,410.070747,0.001224,2.681700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200264,179.857761,310.318012,0.926921,441.430499,187.671603,0.998886,441.231728,187.864689,0.096669,0.076924,...,294.012558,445.905571,0.999984,282.939896,411.287216,0.001462,284.103745,409.418373,0.001374,5134.258045
200265,167.375214,290.504356,0.963188,441.429144,187.634814,0.999135,441.278057,187.812428,0.092426,0.070794,...,293.741898,445.909035,0.999960,282.974686,411.244926,0.001457,284.845581,410.395554,0.001333,5134.283686
200266,167.725983,291.021839,0.996800,441.431580,187.595816,0.999472,441.208755,187.758507,0.100853,0.065544,...,292.409706,447.389526,0.999672,284.155411,411.187981,0.001471,283.402618,410.220284,0.001636,5134.309327
200267,167.725983,291.021839,0.996800,441.433312,187.576316,0.999472,441.208755,187.758507,0.100853,0.065544,...,292.409706,447.389526,0.999672,284.155411,411.187981,0.001471,283.402618,410.220284,0.001636,5134.334968


In [10]:
df_tracks.index.size

200269

In [11]:
set([c.split("_")[0] for c in df_tracks.columns])

{'nose', 'paw', 'pupil', 'times', 'tongue', 'tube'}

In [12]:
keypoints = [
    "nose_tip",
    "pupil_top_r",
    "pupil_bottom_r",
    "pupil_right_r",
    "pupil_left_r",
    "paw_l",
    "paw_r",
    "tongue_end_l",
    "tongue_end_r",
]

keypoints_cols = np.array([(f"{k}_x", f"{k}_y", f"{k}_likelihood") for k in keypoints])
likelihood_cols = keypoints_cols[:, -1]
for k in keypoints_cols.ravel():
    if k not in df_tracks.columns:
        print(k)

In [13]:
keypoints_cols[:, :-1], likelihood_cols

(array([['nose_tip_x', 'nose_tip_y'],
        ['pupil_top_r_x', 'pupil_top_r_y'],
        ['pupil_bottom_r_x', 'pupil_bottom_r_y'],
        ['pupil_right_r_x', 'pupil_right_r_y'],
        ['pupil_left_r_x', 'pupil_left_r_y'],
        ['paw_l_x', 'paw_l_y'],
        ['paw_r_x', 'paw_r_y'],
        ['tongue_end_l_x', 'tongue_end_l_y'],
        ['tongue_end_r_x', 'tongue_end_r_y']], dtype='<U25'),
 array(['nose_tip_likelihood', 'pupil_top_r_likelihood',
        'pupil_bottom_r_likelihood', 'pupil_right_r_likelihood',
        'pupil_left_r_likelihood', 'paw_l_likelihood', 'paw_r_likelihood',
        'tongue_end_l_likelihood', 'tongue_end_r_likelihood'], dtype='<U25'))

In [14]:
df_trials = pd.read_csv(parent_path.joinpath("trials_data.csv"))
df_trials

,goCueTrigger_times,included,quiescencePeriod,stimOff_times,stimOffTrigger_times,stimOnTrigger_times,goCue_times,response_times,choice,stimOn_times,contrastLeft,contrastRight,feedback_times,feedbackType,rewardVolume,probabilityLeft,firstMovement_times
0,3.678400,True,0.693412,6.200200,6.145700,3.578300,3.698300,4.145600,-1.0,3.786400,0.000,NaN,4.175400,-1.0,0.0,0.5,4.086964
1,7.886400,True,0.419460,9.266900,9.221900,7.832500,7.916900,8.221800,1.0,7.886300,0.250,NaN,8.221900,1.0,1.5,0.5,8.145964
2,11.119700,True,0.558785,13.419500,13.371900,11.072700,11.150200,11.371800,1.0,11.119600,NaN,0.1250,11.399300,-1.0,0.0,0.5,11.279964
3,15.733500,True,0.410008,17.052900,17.007200,15.689900,15.760800,16.007100,1.0,15.733400,0.250,NaN,16.007200,1.0,1.5,0.5,15.919964
4,19.316700,True,0.582946,20.636000,20.591800,19.274100,19.346200,19.591700,1.0,19.316600,1.000,NaN,19.591800,1.0,1.5,0.5,19.510964
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
950,5040.976099,False,0.428614,5042.395299,5042.331699,5040.916999,5041.006199,5041.331599,1.0,5040.975999,0.250,NaN,5041.331699,1.0,1.5,0.8,5041.223964
951,5044.309399,False,0.648009,5047.745399,5047.684499,5044.243399,5044.339599,5046.684399,-1.0,5044.309299,NaN,0.0625,5046.684499,1.0,1.5,0.8,5046.492964
952,5049.459298,False,0.440100,5050.842498,5050.786098,5049.394298,5049.489198,5049.785998,1.0,5049.459198,1.000,NaN,5049.786098,1.0,1.5,0.8,5049.668964
953,5053.028698,False,0.427945,5100.510898,5100.458198,5052.974398,5053.058698,5098.458098,-1.0,5053.028598,0.000,NaN,5098.489198,-1.0,0.0,0.8,5098.376964


In [15]:
from fastplotlib.widgets.nd_widget import NDPositions, ndp_extras, NDImage

# Create the ND objects

### NDImage from the video, provide it the `time (sec) -> array index` mapping for the first dimension

In [16]:
ndi = NDImage(
    vid, 
    index_mappings=(vid_second_to_index,), 
    processor_kwargs={"compute_histogram": False, "rgb": True}
)

# NDPositions object to view the tracks and likelihood

In [17]:
ndp_scatters = NDPositions(
    df_tracks,  # tracks dataframe
    keypoints_cols[:, :-1],  # provide the columns to take the tracks data from in the form [(keypoint_1_x, keypoint_1_y), ... (keypoint_n_x), (keypoint_n_y)}, 
    likelihood_cols,  # show likelihood as tooltips
    processor=ndp_extras.NDPP_Pandas, # use the processor for pandas dataframes
    graphic=fpl.ScatterCollection, # represent graphically as a scatter collection
    display_window=5,
    index_mappings=(tracks_second_to_index,)  # time (s) -> array index mapping
)

ndp_lines = NDPositions(
    df_tracks,
    keypoints_cols[:, :-1],
    processor=ndp_extras.NDPP_Pandas, 
    graphic=fpl.LineCollection, 
    display_window=5,
    index_mappings=(tracks_second_to_index,)
)

# likelihood as a heatmap
ndp_ll = NDPositions(
    df_tracks,
    [("times", c) for c in likelihood_cols],  # must provide the "times" x-values!
    processor=ndp_extras.NDPP_Pandas, 
    graphic=fpl.ImageGraphic, 
    display_window=5,
    index_mappings=(tracks_second_to_index,)
)

In [18]:
# UI elements for notebook use
from ipywidgets import FloatSlider, VBox, Layout

In [19]:
# list of the arrays we want to update
nd_arrays = [ndi, ndp_scatters, ndp_lines, ndp_ll]

### Sliders and Functions to update the ND objects when the sliders move

In [20]:
def update_dw(change):
    for ndp in nd_arrays:
        ndp.display_window = change["new"]
    fig[0, 1].auto_scale()

slider_dw = FloatSlider(
    min=0, max=30.0, 
    value=df_tracks["times"][0], description="disp window", layout=Layout(width="600px")
)

slider_dw.observe(update_dw, "value")

def update_t(change):
    for ndp in nd_arrays:
        ndp.indices = (change["new"],)
    fig[0, 1].auto_scale()

slider = FloatSlider(
    min=df_tracks["times"][0], max=df_tracks["times"].values[-1], value=0, 
    description="index", layout=Layout(width="600px")
)
slider.observe(update_t, "value")

### Create Figure, Add the graphics to the subplots

In [21]:
fig = fpl.Figure(shape=(1, 2), size=(900, 400))

fig[0, 0].add_graphic(ndi.graphic)
fig[0, 0].add_graphic(ndp_scatters.graphic)
fig[0, 0].add_graphic(ndp_lines.graphic)
fig[0, 1].add_graphic(ndp_ll.graphic)
fig[0, 1].camera.maintain_aspect = False


VBox([fig.show(), slider_dw, slider])

RFBOutputContext()

In [23]:
# a cmap for the scatter collection, each keypoint will get its own color
# There is a lot of fine-tuning you can do for scatter colors
ndp_scatters.graphic.cmap = "tab10"

# change some properties of the scatter
for g in ndp_scatters.graphic:
    g.sizes = 5
    g.visible = True
    g.edge_width = 0.5

In [24]:
# hide the line tracks for now
for g in ndp_lines.graphic:
    g.color_mode = "vertex"
    g.cmap = "viridis"
    g.visible = False

In [30]:
# You can change the graphical representations at any time
# ndp_ll.graphic = fpl.LineStack

In [31]:
# reset vmin, vmax of heatmap if necessary
ndp_ll.graphic.reset_vmin_vmax()